In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans

Loading Datasets

In [3]:
df1 = pd.read_csv("../data/Audible_Catlog.csv")
df2 = pd.read_csv("../data/Audible_Catlog_Advanced_Features.csv")

In [4]:
df1.shape

(6368, 5)

In [5]:
df2.shape

(4464, 8)

In [14]:
df = pd.merge(df1, df2, on=["Book Name","Author"], how="inner",  suffixes=('', '_df2'))
df

,Book Name,Author,Rating,Number of Reviews,Price,Rating_df2,Number of Reviews_df2,Price_df2,Description,Listening Time,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313.0,10080.0,4.9,371.0,10080,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658.0,615.0,4.6,3682.0,615,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174.0,10378.0,4.4,20306.0,10378,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4614.0,888.0,4.6,4678.0,888,Brought to you by Penguin.,5 hours and 35 minutes,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4302.0,1005.0,4.6,4308.0,1005,"Stop going through life, Start growing throug...",6 hours and 25 minutes,",#6 in Audible Audiobooks & Originals (See Top..."
...,...,...,...,...,...,...,...,...,...,...,...
4246,The Prophet & The Wanderer,Khalil Gibran,4.1,6.0,539.0,4.1,6.0,539,"Sorry, we just need to make sure you're not a ...",-1,-1
4247,Make Today Count: The Secret of Your Success I...,John C. Maxwell,4.7,301.0,500.0,4.7,307.0,500,"Sorry, we just need to make sure you're not a ...",-1,-1
4248,Make Today Count: The Secret of Your Success I...,John C. Maxwell,4.7,301.0,500.0,4.7,307.0,500,"Sorry, we just need to make sure you're not a ...",-1,-1
4249,वॉयर - एक कामुक लघुकथा,सेसिलि रोसडैल,-1.0,NaN,65.0,-1.0,NaN,65,"Sorry, we just need to make sure you're not a ...",-1,-1


In [15]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4251 entries, 0 to 4250
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Book Name              4251 non-null   object 
 1   Author                 4251 non-null   object 
 2   Rating                 4251 non-null   float64
 3   Number of Reviews      3838 non-null   float64
 4   Price                  4249 non-null   float64
 5   Rating_df2             4251 non-null   float64
 6   Number of Reviews_df2  3839 non-null   float64
 7   Price_df2              4251 non-null   int64  
 8   Description            4245 non-null   object 
 9   Listening Time         4251 non-null   object 
 10  Ranks and Genre        4251 non-null   object 
dtypes: float64(5), int64(1), object(5)
memory usage: 365.4+ KB


In [16]:
df.columns


Index(['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price',
       'Rating_df2', 'Number of Reviews_df2', 'Price_df2', 'Description',
       'Listening Time', 'Ranks and Genre'],
      dtype='object')

In [17]:
df.duplicated(subset=['Book Name', 'Author']).sum()
duplicates = df[df.duplicated(subset=['Book Name', 'Author'], keep=False)]
print(duplicates)

                                              Book Name             Author  \
6                                               Sapiens  Yuval Noah Harari   
7                                               Sapiens  Yuval Noah Harari   
46                                  Think and Grow Rich      Napoleon Hill   
47                                  Think and Grow Rich      Napoleon Hill   
80                                              Sapiens  Yuval Noah Harari   
...                                                 ...                ...   
4246                         The Prophet & The Wanderer      Khalil Gibran   
4247  Make Today Count: The Secret of Your Success I...    John C. Maxwell   
4248  Make Today Count: The Secret of Your Success I...    John C. Maxwell   
4249                             वॉयर - एक कामुक लघुकथा      सेसिलि रोसडैल   
4250                             वॉयर - एक कामुक लघुकथा      सेसिलि रोसडैल   

      Rating  Number of Reviews   Price  Rating_df2  Number of 

In [18]:
df = df.drop_duplicates(subset=['Book Name', 'Author'])

In [19]:
df.duplicated(subset=['Book Name', 'Author']).sum()

np.int64(0)

In [21]:
print(df.shape)

(3351, 11)


In [22]:
df.columns

Index(['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price',
       'Rating_df2', 'Number of Reviews_df2', 'Price_df2', 'Description',
       'Listening Time', 'Ranks and Genre'],
      dtype='object')

In [24]:


def consolidate_duplicate_columns(df):

    df_consolidated = df.copy()
    df2_cols = [col for col in df_consolidated.columns if col.endswith('_df2')]
    
    for df2_col in df2_cols:
        # Get original column name (without _df2)
        original_col = df2_col.replace('_df2', '')
        
        if original_col in df_consolidated.columns:
            df_consolidated[original_col].fillna(df_consolidated[df2_col], inplace=True)
            df_consolidated.drop(columns=[df2_col], inplace=True)
            print(f"Consolidated: {original_col}")
    return df_consolidated

df_final = consolidate_duplicate_columns(df)

print(f"\n✅ Final dataset shape: {df_final.shape}")
print(f"Final columns: {df_final.columns.tolist()}")

Consolidated: Rating
Consolidated: Number of Reviews
Consolidated: Price

✅ Final dataset shape: (3351, 8)
Final columns: ['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre']


C:\Users\S0431440\AppData\Local\Temp\ipykernel_27276\2472253439.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_consolidated[original_col].fillna(df_consolidated[df2_col], inplace=True)
C:\Users\S0431440\AppData\Local\Temp\ipykernel_27276\2472253439.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

In [34]:
df_final.isnull().sum()

Book Name              0
Author                 0
Rating                 0
Number of Reviews    320
Price                  0
Description            5
Listening Time         0
Ranks and Genre        0
dtype: int64

In [41]:


df_final[df_final['Number of Reviews'].isnull()]

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre
55,FREE: Professional Integrity (A Riyria Chronic...,Michael J. Sullivan,-1.0,NaN,0.0,\n\nOops!\nIt's rush hour and traffic is pilin...,-1,-1
123,Flow: Living at the Peak of Your Abilities,Mihaly Csikszentmihalyi Ph.D.,-1.0,NaN,1439.0,"In flow, everyday experience becomes a moment ...",5 hours and 31 minutes,",#180 in Audible Audiobooks & Originals (See T..."
134,Kyon Se Shuruwat Karein [Start with Why]: Maha...,Simon Sinek,-1.0,NaN,836.0,Why are some people and organizations more inv...,10 hours and 52 minutes,",#153 in Audible Audiobooks & Originals (See T..."
150,H.G. Wells: The Science Fiction Collection,H. G. Wells,-1.0,NaN,1898.0,He’s often been called the father of science f...,27 hours and 15 minutes,",#237 in Audible Audiobooks & Originals (See T..."
160,20 Bedtime Stories For Kids,div.,-1.0,NaN,414.0,"Sorry, we just need to make sure you're not a ...",-1,-1
...,...,...,...,...,...,...,...,...
4179,Break Through Pain: How to Relieve Pain Using ...,Shinzen Young,-1.0,NaN,492.0,"Sorry, we just need to make sure you're not a ...",-1,-1
4200,Küçük Prens,Antoine De Saint-Exupery,-1.0,NaN,300.0,Gezegenindeki çiçeğiyle pek anlaşamadığı için ...,-1,-1
4217,Colin Morgan: Audible Sessions: FREE Exclusive...,Robin Morgan-Bentley,-1.0,NaN,0.0,"Sorry, we just need to make sure you're not a ...",-1,-1
4230,वॉयर - एक कामुक लघुकथा,सेसिलि रोसडैल,-1.0,NaN,65.0,"Sorry, we just need to make sure you're not a ...",-1,-1


In [42]:
df_final.dropna(subset=['Number of Reviews'], inplace=True)

In [43]:
df_final

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313.0,10080.0,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658.0,615.0,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174.0,10378.0,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4614.0,888.0,Brought to you by Penguin.,5 hours and 35 minutes,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4302.0,1005.0,"Stop going through life, Start growing throug...",6 hours and 25 minutes,",#6 in Audible Audiobooks & Originals (See Top..."
...,...,...,...,...,...,...,...,...
4224,The Death of WCW,R.D. Reynolds,4.6,271.0,836.0,"Sorry, we just need to make sure you're not a ...",-1,-1
4226,The Prophet & The Wanderer,Khalil Gibran,4.1,6.0,539.0,"Sorry, we just need to make sure you're not a ...",-1,-1
4228,Make Today Count: The Secret of Your Success I...,John C. Maxwell,4.7,301.0,500.0,"Sorry, we just need to make sure you're not a ...",-1,-1
4234,"Kill the Company: End the Status Quo, Start an...",Lisa Bodell,4.2,26.0,586.0,\n\nOops!\nIt's rush hour and traffic is pilin...,-1,-1


In [44]:
# Reset index for clean indexing
df_final = df_final.reset_index(drop=True)

# Final check for missing values
print("Final Missing Values Summary:")
missing_final = pd.DataFrame({
    'Column': df_final.columns,
    'Missing_Count': df_final.isnull().sum().values,
    'Missing_Percentage': (df_final.isnull().sum().values / len(df_final) * 100).round(2)
})
display(missing_final[missing_final['Missing_Count'] > 0])

print(f"\n✅ Data cleaning and merging complete!")
print(f"Total books in final dataset: {len(df_final)}")

Final Missing Values Summary:


,Column,Missing_Count,Missing_Percentage
5,Description,4,0.13



✅ Data cleaning and merging complete!
Total books in final dataset: 3031


In [50]:
df_final.drop(df_final[df_final['Rating'] <= 0].index, inplace=True)

In [52]:
df_final.describe()

,Rating,Number of Reviews,Price
count,3030.000000,3030.000000,3030.000000
mean,4.452112,1164.229703,1003.023102
std,0.354093,3082.218892,1807.155828
min,1.000000,1.000000,0.000000
25%,4.300000,76.250000,516.000000
50%,4.500000,284.000000,683.000000
75%,4.700000,968.000000,888.000000
max,5.000000,70077.000000,18290.000000
